<a href="https://colab.research.google.com/github/charles-shaju/sku110k-dense-detection/blob/main/Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Phase 1: Setup & Installations

In [ ]:
%pip install -q roboflow ultralytics pyyaml gradio onnx onnxslim

In [ ]:
import random
import shutil
import time
from pathlib import Path
import cv2
import numpy as np
import yaml
from google.colab import userdata
from roboflow import Roboflow
from ultralytics import YOLO
import gradio as gr

### Phase 2: Data Acquisition & Subset Sampling

In [ ]:
# Fetch full dataset from Roboflow
api_key = userdata.get("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)
project = rf.workspace("jacobs-workspace").project("sku-110k")
dataset = project.version(6).download(model_format="yolov8", location="/content/sku-110k-full")

In [ ]:
# Create 1,000 image subset split
SOURCE = Path(dataset.location)
OUTPUT = Path("/content/sku-110k-1000")

SPLIT_COUNTS = {
    "train": 700,
    "valid": 200,
    "test": 100,
}

random.seed(42)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

for split, requested_count in SPLIT_COUNTS.items():
    source_images = SOURCE / split / "images"
    source_labels = SOURCE / split / "labels"
    output_images = OUTPUT / split / "images"
    output_labels = OUTPUT / split / "labels"

    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    images = [p for p in source_images.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    selected = random.sample(images, min(requested_count, len(images)))

    for img_path in selected:
        shutil.copy2(img_path, output_images / img_path.name)
        lbl_path = source_labels / f"{img_path.stem}.txt"
        if lbl_path.exists():
            shutil.copy2(lbl_path, output_labels / lbl_path.name)

    print(f"{split}: copied {len(selected)} images.")

In [ ]:
# Generate data.yaml configuration file
with open(SOURCE / "data.yaml", "r") as f:
    source_config = yaml.safe_load(f)

subset_config = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": source_config["names"],
}
if "nc" in source_config:
    subset_config["nc"] = source_config["nc"]

with open(OUTPUT / "data.yaml", "w") as f:
    yaml.safe_dump(subset_config, f, sort_keys=False)

print("Config generated at:", OUTPUT / "data.yaml")

### Phase 3: Model Training

In [ ]:
# Train YOLO11n on the subset
model = YOLO("yolo11n.pt")
results = model.train(
    data=str(OUTPUT / "data.yaml"),
    epochs=50,
    imgsz=640,
    max_det=533,
    batch=16,
    project="/content/runs",
    name="sku110k-yolov8"
)

### Phase 4: Model Exporting

In [ ]:
# Export PyTorch model to ONNX & TensorRT formats
best_weights = "/content/runs/sku110k-yolov8/weights/best.pt"
model_eval = YOLO(best_weights)

# Export to ONNX
model_eval.export(format='onnx', imgsz=640, simplify=True)

# Export to TensorRT Engine
model_eval.export(format='engine', imgsz=640, half=True)

### Phase 5: Benchmarking & Deployment UI

In [ ]:
def benchmark(model_path, label, n_runs=30, warmup=5):
    bm_model = YOLO(model_path)
    test_img = next(Path("/content/sku-110k-1000/test/images").glob("*.jpg"))

    # Warmup runs
    for _ in range(warmup):
        bm_model.predict(test_img, verbose=False)

    times = []
    for _ in range(n_runs):
        t0 = time.time()
        bm_model.predict(test_img, verbose=False)
        times.append((time.time() - t0) * 1000)

    avg_t = sum(times) / len(times)
    print(f"{label}: {avg_t:.1f}ms avg (min {min(times):.1f}ms, max {max(times):.1f}ms)")
    return avg_t

# Run the benchmarks
pt_time = benchmark('/content/runs/sku110k-yolov8/weights/best.pt', 'PyTorch (.pt)')
onnx_time = benchmark('/content/runs/sku110k-yolov8/weights/best.onnx', 'ONNX')
engine_time = benchmark('/content/runs/sku110k-yolov8/weights/best.engine', 'TensorRT (.engine)')

In [ ]:
# Gradio Interface Deployment
demo_model = YOLO('/content/runs/sku110k-yolov8/weights/best.onnx')

def detect(image):
    start = time.time()
    results = demo_model.predict(image, conf=0.25, verbose=False)
    latency_ms = (time.time() - start) * 1000

    result = results[0]
    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    n_objects = len(result.boxes)
    caption = f"Detected {n_objects} objects in {latency_ms:.1f}ms (ONNX Model)"
    return annotated, caption

demo = gr.Interface(
    fn=detect,
    inputs=gr.Image(type="numpy", label="Upload shelf/warehouse image"),
    outputs=[
        gr.Image(type="numpy", label="Detections"),
        gr.Textbox(label="Result")
    ],
    title="Dense Object Detection — SKU-110K (YOLO11)",
    description="Deployment model trained on SKU-110K dense retail shelf data for quick inventory tracking."
)

if __name__ == "__main__":
    demo.launch(share=True)